# TSFM4NILM : Sequence-to-Point Energy Disaggregation Using MOMENT

## EDA and Splits

### Cleaned Data source : https://pureportal.strath.ac.uk/en/datasets/refit-electrical-load-measurements-cleaned/

### Cleaned Dataset Specifics :

**Naming scheme:** House 1 to House 21 (without House 14).

**Dataset format:** DateTime (YYYY-MM-DD HH:mm:ss), UNIX TIMESTAMP (UCT), Aggregate, Appliance1, Appliance2, ... , Appliance9.

**Missing data:** Forward filled NaN values and remove values which exceed 4000 Watts (maximum possible power draw). 

**Solar planel:** House 11 and 21 aggregates are affected by solar panels, dropping them.

**Issues(boolean):** Rows where the sum of appliance powers > aggregate power. To be handled







## Data Loading

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("Data")

# Individual appliance monitors (IAMs) mapped to corresponding appliance per house
appliance_map = {
    1:  {0: "Aggregate", 1: "Fridge", 2: "Chest Freezer", 3: "Upright Freezer", 4: "Tumble Dryer",
         5: "Washing Machine", 6: "Dishwasher", 7: "Computer Site", 8: "Television Site", 9: "Electric Heater"},
    2:  {0: "Aggregate", 1: "Fridge-Freezer", 2: "Washing Machine", 3: "Dishwasher", 4: "Television",
         5: "Microwave", 6: "Toaster", 7: "Hi-Fi", 8: "Kettle", 9: "Oven Extractor Fan"},
    3:  {0: "Aggregate", 1: "Toaster", 2: "Fridge-Freezer", 3: "Freezer", 4: "Tumble Dryer",
         5: "Dishwasher", 6: "Washing Machine", 7: "Television", 8: "Microwave", 9: "Kettle"},
    4:  {0: "Aggregate", 1: "Fridge", 2: "Freezer", 3: "Fridge-Freezer", 4: "Washing Machine (1)",
         5: "Washing Machine (2)", 6: "Computer Site", 7: "Television Site", 8: "Microwave", 9: "Kettle"},
    5:  {0: "Aggregate", 1: "Fridge-Freezer", 2: "Tumble Dryer", 3: "Washing Machine", 4: "Dishwasher",
         5: "Computer Site", 6: "Television Site", 7: "Combination Microwave", 8: "Kettle", 9: "Toaster"},
    6:  {0: "Aggregate", 1: "Freezer (Utility Room)", 2: "Washing Machine", 3: "Dishwasher", 4: "MJY Computer",
         5: "Television Site", 6: "Microwave", 7: "Kettle", 8: "Toaster", 9: "PGM Computer"},
    7:  {0: "Aggregate", 1: "Fridge", 2: "Freezer (Garage)", 3: "Freezer", 4: "Tumble Dryer",
         5: "Washing Machine", 6: "Dishwasher", 7: "Television Site", 8: "Toaster", 9: "Kettle"},
    8:  {0: "Aggregate", 1: "Fridge", 2: "Freezer", 3: "Dryer", 4: "Washing Machine",
         5: "Toaster", 6: "Computer", 7: "Television Site", 8: "Microwave", 9: "Kettle"},
    9:  {0: "Aggregate", 1: "Fridge-Freezer", 2: "Washer Dryer", 3: "Washing Machine", 4: "Dishwasher",
         5: "Television Site", 6: "Microwave", 7: "Kettle", 8: "Hi-Fi", 9: "Electric Heater"},
    10: {0: "Aggregate", 1: "Magimix (Blender)", 2: "Freezer", 3: "Chest Freezer (In Garage)", 4: "Fridge-Freezer",
         5: "Washing Machine", 6: "Dishwasher", 7: "Television Site", 8: "Microwave", 9: "Kenwood KMix"},
    11: {0: "Aggregate", 1: "Fridge", 2: "Fridge-Freezer", 3: "Washing Machine", 4: "Dishwasher",
         5: "Computer Site", 6: "Microwave", 7: "Kettle", 8: "Router", 9: "Hi-Fi"},
    12: {0: "Aggregate", 1: "Fridge-Freezer", 2: "Television Site (Lounge)", 3: "Microwave", 4: "Kettle",
         5: "Toaster", 6: "Television Site (Bedroom)", 7: "Not Used", 8: "Not Used", 9: "Not Used"},
    13: {0: "Aggregate", 1: "Television Site", 2: "Unknown", 3: "Washing Machine", 4: "Dishwasher",
         5: "Tumble Dryer", 6: "Television Site", 7: "Computer Site", 8: "Microwave", 9: "Kettle"},
    15: {0: "Aggregate", 1: "Fridge-Freezer", 2: "Tumble Dryer", 3: "Washing Machine", 4: "Dishwasher",
         5: "Computer Site", 6: "Television Site", 7: "Microwave", 8: "Kettle", 9: "Toaster"},
    16: {0: "Aggregate", 1: "Fridge-Freezer (1)", 2: "Fridge-Freezer (2)", 3: "Electric Heater (1)",
         4: "Electric Heater (2)", 5: "Washing Machine", 6: "Dishwasher", 7: "Computer Site",
         8: "Television Site", 9: "Dehumidifier/Heater"},
    17: {0: "Aggregate", 1: "Freezer (Garage)", 2: "Fridge-Freezer", 3: "Tumble Dryer (Garage)", 4: "Washing Machine",
         5: "Computer Site", 6: "Television Site", 7: "Microwave", 8: "Kettle", 9: "Plug Site (Bedroom)"},
    18: {0: "Aggregate", 1: "Fridge (Garage)", 2: "Freezer (Garage)", 3: "Fridge-Freezer",
         4: "Washer Dryer (Garage)", 5: "Washing Machine", 6: "Dishwasher", 7: "Desktop Computer",
         8: "Television Site", 9: "Microwave"},
    19: {0: "Aggregate", 1: "Fridge & Freezer", 2: "Washing Machine", 3: "Television Site", 4: "Microwave",
         5: "Kettle", 6: "Toaster", 7: "Bread-maker", 8: "Lamp (80Watts)", 9: "Hi-Fi"},
    20: {0: "Aggregate", 1: "Fridge", 2: "Freezer", 3: "Tumble Dryer", 4: "Washing Machine",
         5: "Dishwasher", 6: "Computer Site", 7: "Television Site", 8: "Microwave", 9: "Kettle"},
    21: {0: "Aggregate", 1: "Fridge-Freezer", 2: "Tumble Dryer", 3: "Washing Machine", 4: "Dishwasher",
         5: "Food Mixer", 6: "Television", 7: "Kettle/Toaster", 8: "Vivarium", 9: "Pond Pump"},
}


# Maps IAM numbers to actual appliance names 
# Repeated column names get suffix 1,2,.. etc.

def column_map(house_id):
    names = appliance_map[house_id]
    counts = pd.Series(list(names.values())).value_counts()
    mapping, seen = {}, {}
    for idx, name in names.items():
        if counts[name] > 1:
            seen[name] = seen.get(name, 0) + 1
            name = f"{name} ({seen[name]})"
        mapping["Aggregate" if idx == 0 else f"Appliance{idx}"] = name
    return mapping


# Function to load house by number/id
def load_house(house_id):
    df = pd.read_csv(DATA_DIR / f"CLEAN_House{house_id}.csv", parse_dates=["Time"])
    df = df.rename(columns=column_map(house_id))
    df.insert(0, "House", house_id) # Adds house column with hosue_id at the 0th column index
    return df

# Loads data for each house
house_data_map={}
for house_number in appliance_map:
    house_data_map[house_number]=load_house(house_number)

## Data Summary

In [46]:
def data_summary(house_data, gap_threshold=pd.Timedelta("2min")):
    house_id = house_data["House"].iloc[0] # Selects the first row i.e house number/id

    # Number of rows with issues=1 
    issues = int((house_data["Issues"] == 1).sum())

    time = house_data["Time"]
    is_sorted = time.is_monotonic_increasing #Check if timestamps are sorted (boolean)
    deltas = time.sort_values().diff().dropna() #Time difference between subsequent timestamps as a list
    sampling_interval = deltas.median() # Typical delta
    # Intervals longer than gap_threshold count as gaps (matches the 2-min forward-fill cutoff used in cleaning)
    gaps = deltas[deltas > gap_threshold]

    print(f"House {house_id}")
    print(f"Rows: {len(house_data):,}")
    print(f"Columns ({house_data.shape[1]}): {list(house_data.columns)}")
    print(f"Issues: {issues:,} rows ({issues / len(house_data):.2%})")
    print(f"Start: {time.min()}  End: {time.max()}  Span: {time.max() - time.min()}")
    print(f"Timestamps sorted: {is_sorted}")
    print(f"Sampling interval (median): {sampling_interval}")
    print(f"Gaps > {gap_threshold}: {len(gaps):,}"
          + (f"  (longest: {gaps.max()}, total missing time: {gaps.sum()})" if len(gaps) else ""))
    print("-" * 50)

    return {"House": house_id, "Rows": len(house_data), "Columns": list(house_data.columns),
            "Issues": issues,
            "Start": time.min(), "End": time.max(),
            "Sorted": is_sorted, "Sampling Interval": sampling_interval,
            "Gaps": len(gaps), "Longest Gap": gaps.max() if len(gaps) else pd.Timedelta(0)}

summaries = [data_summary(df) for df in house_data_map.values()]

House 1
Rows: 6,960,008
Columns (14): ['House', 'Time', 'Unix', 'Aggregate', 'Fridge', 'Chest Freezer', 'Upright Freezer', 'Tumble Dryer', 'Washing Machine', 'Dishwasher', 'Computer Site', 'Television Site', 'Electric Heater', 'Issues']
Issues: 58,183 rows (0.84%)
Start: 2013-10-09 13:06:17  End: 2015-07-10 11:56:32  Span: 638 days 22:50:15
Timestamps sorted: True
Sampling interval (median): 0 days 00:00:07
Gaps > 0 days 00:02:00: 659  (longest: 41 days 14:26:53, total missing time: 80 days 23:53:16)
--------------------------------------------------
House 2
Rows: 5,733,526
Columns (14): ['House', 'Time', 'Unix', 'Aggregate', 'Fridge-Freezer', 'Washing Machine', 'Dishwasher', 'Television', 'Microwave', 'Toaster', 'Hi-Fi', 'Kettle', 'Oven Extractor Fan', 'Issues']
Issues: 28,444 rows (0.50%)
Start: 2013-09-17 22:08:11  End: 2015-05-28 08:05:43  Span: 617 days 09:57:32
Timestamps sorted: True
Sampling interval (median): 0 days 00:00:07
Gaps > 0 days 00:02:00: 597  (longest: 62 days 00:35

### Findings 

1. The cleaned dataset's claim of 'zeroed (> 2 minute gaps)' isn't true as there are existing gaps with a 2min delta. This needs to be handled by segmenting the data such that no gaps existing within input windows.

2. There are significant number of rows with sum of appliance power > aggregate power (issues=1) which needs to be handled.

## Handling Issue=1

### Issue runs : subsequent issue rows > 2mins

Tells us if the issue rows are scattered or continous and how much data loss can we expect if they are to be removed.

In [50]:
def issue_runs(house_data, gap_threshold=pd.Timedelta("2min")):
    time = house_data["Time"]
    is_issue = house_data["Issues"] == 1

    # A new run starts when Issues flips, or when a >2min time gap breaks the stretch
    # (a gap already ends the segment, so it shouldn't stretch a run's duration)
    new_run = (is_issue != is_issue.shift()) | (time.diff() > gap_threshold)
    run_id = new_run.cumsum()

    runs = (house_data[is_issue]
            .groupby(run_id[is_issue])["Time"]
            .agg(start="min", end="max", rows="size"))
    runs["duration"] = runs["end"] - runs["start"]
    return runs


LONG_RUN = pd.Timedelta("2min")

rows = {}
for house_id, house_data in house_data_map.items():
    if house_id in (11, 21):  # solar houses, dropped
        continue
    runs = issue_runs(house_data)
    long_runs = runs[runs["duration"] > LONG_RUN]
    rows[house_id] = {
        "Issue runs": len(runs),
        "Runs > 2min": len(long_runs),
        "Longest run": runs["duration"].max(),
        "Rows removed": int(long_runs["rows"].sum()),
        "% of issue rows": long_runs["rows"].sum() / runs["rows"].sum(),
        "% of house rows": long_runs["rows"].sum() / len(house_data),
        "Time removed": long_runs["duration"].sum(),
    }

long_run_summary = pd.DataFrame.from_dict(rows, orient="index")

total_rows = sum(len(house_data_map[h]) for h in long_run_summary.index)
total_issue_rows = sum(int((house_data_map[h]["Issues"] == 1).sum()) for h in long_run_summary.index)
print(f"Total runs > 2min: {long_run_summary['Runs > 2min'].sum():,}")
print(f"Total rows removed: {long_run_summary['Rows removed'].sum():,} "
      f"({long_run_summary['Rows removed'].sum() / total_issue_rows:.2%} of issue rows, "
      f"{long_run_summary['Rows removed'].sum() / total_rows:.2%} of all rows)")
print(f"Total time removed: {long_run_summary['Time removed'].sum()}")

long_run_summary.style.format({"% of issue rows": "{:.2%}", "% of house rows": "{:.2%}",
                               "Rows removed": "{:,}", "Issue runs": "{:,}"})

Total runs > 2min: 12,440
Total rows removed: 1,149,892 (64.21% of issue rows, 1.05% of all rows)
Total time removed: 89 days 01:11:28


,Issue runs,Runs > 2min,Longest run,Rows removed,% of issue rows,% of house rows,Time removed
1,"13,504",394,0 days 02:26:46,"33,907",58.28%,0.49%,2 days 14:17:14
2,"13,877",47,0 days 04:01:55,"6,573",23.11%,0.11%,0 days 12:15:15
3,"26,430",2388,0 days 09:51:14,"344,942",84.41%,4.93%,25 days 03:49:21
4,"22,193",303,0 days 00:55:32,"32,502",48.19%,0.48%,2 days 08:35:36
5,"35,360",3425,1 days 20:33:28,"325,645",76.48%,4.38%,25 days 21:27:40
6,"13,744",298,0 days 00:23:48,"6,323",18.35%,0.10%,0 days 11:17:16
7,"29,320",2516,0 days 00:24:32,"99,411",61.40%,1.47%,7 days 10:08:50
8,"12,510",55,0 days 00:35:59,"3,352",13.41%,0.05%,0 days 06:49:56
9,"18,674",48,0 days 00:47:37,"3,655",11.34%,0.06%,0 days 06:08:12
10,"15,730",100,0 days 00:42:27,"6,659",22.08%,0.10%,0 days 11:48:50


Based on the above table, Two tier handling approach :
1. Mask short runs/scattered issue rows i.e discard window if anchor point for target appliance > aggregate value.

2. For long issue runs, remove rows and apply segmentation. (minimal data loss ~ few rows, therefore affordable).

Exceotion : Five houses show significant row removals if the above approach is followed : In houses 3, 5, 7, 13 and 18, 61–84% of issue rows fall in long runs, which points to a systematic cause rather than timing glitches. These need investigation before applying the above rules.

### Excess profiles for the exception houses

Excess = sum(appliances) - aggregate

Tells us the possible calibration issue along with any dead aggregates across issue runs.

In [ ]:
def long_run_profile(df, min_dur_s=120):
    app_cols = [c for c in df.columns if c not in ("House", "Time", "Unix", "Aggregate", "Issues")]

    issue = df["Issues"] == 1
    gap = df["Unix"].diff() > 120
    run_id = ((issue != issue.shift()) | gap).cumsum()   # new run at value change OR gap

    r = df[issue].assign(run=run_id[issue],
                         excess=df[app_cols].sum(axis=1) - df["Aggregate"])
    runs = r.groupby("run").agg(start=("Unix", "min"), end=("Unix", "max"),
                                median_excess=("excess", "median"),
                                agg_std=("Aggregate", "std"),
                                agg_median=("Aggregate", "median"))
    runs["dur_s"] = runs.end - runs.start
    long = runs[runs.dur_s > min_dur_s].copy()

    # Keep intermediates for the cause checks in the next cell
    return {"df": df, "app_cols": app_cols, "issue": issue, "run_id": run_id, "r": r, "long": long}


profiles = {}
for house_id in [3, 5, 7, 13, 18]:
    profiles[house_id] = long_run_profile(house_data_map[house_id])
    print(f"House {house_id}")
    display(profiles[house_id]["long"][["median_excess", "agg_std", "agg_median"]].describe().round(1))   # per-house summary

House 3


,median_excess,agg_std,agg_median
count,2388.0,2388.0,2388.0
mean,607.1,59.6,485.2
std,824.1,176.7,597.1
min,2.0,0.0,0.0
25%,41.0,0.0,165.0
50%,111.0,3.2,221.0
75%,1222.9,28.9,473.0
max,4317.0,1618.2,4182.0


House 5


,median_excess,agg_std,agg_median
count,3425.0,3425.0,3425.0
mean,358.0,198.8,1526.0
std,450.7,256.3,1020.2
min,3.0,1.1,156.0
25%,46.0,14.8,431.0
50%,206.0,77.7,1916.5
75%,344.5,283.4,2260.5
max,3019.0,2298.7,7443.5


House 7


,median_excess,agg_std,agg_median
count,2516.0,2516.0,2516.0
mean,626.5,354.4,2488.6
std,405.7,329.9,1039.3
min,2.0,0.0,0.0
25%,368.8,69.6,1743.0
50%,565.5,280.4,2682.8
75%,746.0,532.2,2953.1
max,2996.0,2113.1,5425.0


House 13


,median_excess,agg_std,agg_median
count,898.0,898.0,898.0
mean,246.1,39.4,274.5
std,619.3,153.8,395.6
min,1.0,0.0,91.0
25%,8.0,1.2,130.0
50%,42.0,2.5,147.5
75%,50.5,8.7,275.0
max,2461.5,1565.4,4252.0


House 18


,median_excess,agg_std,agg_median
count,1150.0,1150.0,1150.0
mean,86.0,18.8,214.6
std,221.2,28.1,164.4
min,1.0,0.0,0.0
25%,19.0,0.0,142.0
50%,41.0,2.9,208.5
75%,89.9,36.7,268.0
max,2347.5,220.0,3342.0


- 13 and 18 : The median excess is about 41–42 W which points to some sort of calibration or offset error in the plug monitors rather than broken data.

- 5 and 7 : The median excess is about 206 W (house 5, roughly 10% of load) and about 566 W (house 7, roughly 20%), while the aggregate is about 1.9–2.7 kW and actively varying. That points to a load-dependent mismatch between the sub-meters and the mains meter.

- 3 and 18 : At least 25% of their long runs have a perfectly flat aggregate (std = 0), and some read 0 W. The mains meter stopped reporting while appliances kept running, which also explains house 3’s very large excesses (75th percentile about 1.2 kW).

In [54]:
for house_id, p in profiles.items():
    df, app_cols, issue, run_id, long = (p[k] for k in ("df", "app_cols", "issue", "run_id", "long"))
    print(f"House {house_id}")

    # 1. Frozen/dropped aggregate (cause C)
    long["frozen"] = (long.agg_std < 0.5) | (long.agg_median < 10)
    print(f"Share of long runs with a dead aggregate: {long.frozen.mean():.2%}")

    # 2. Which channel dominates? (cause B)
    ch = df[issue].groupby(run_id[issue])[app_cols].mean().loc[long.index]
    print("Dominant channel per long run:")
    display(ch.idxmax(axis=1).value_counts())       # one channel winning most runs = faulty sub-meter
    print("-" * 50)

House 3
Share of long runs with a dead aggregate: 46.19%
Dominant channel per long run:


Television         835
Tumble Dryer       415
Kettle             315
Fridge-Freezer     305
Washing Machine    216
Dishwasher         183
Freezer             88
Microwave           21
Toaster             10
Name: count, dtype: int64

--------------------------------------------------
House 5
Share of long runs with a dead aggregate: 0.00%
Dominant channel per long run:


Dishwasher               1400
Tumble Dryer              885
Fridge-Freezer            751
Washing Machine           215
Kettle                     86
Toaster                    41
Combination Microwave      34
Computer Site              12
Television Site             1
Name: count, dtype: int64

--------------------------------------------------
House 7
Share of long runs with a dead aggregate: 0.83%
Dominant channel per long run:


...

--------------------------------------------------
House 13
Share of long runs with a dead aggregate: 3.56%
Dominant channel per long run:


Tumble Dryer           708
Unknown                 78
Television Site (2)     62
Kettle                  20
Television Site (1)     13
Washing Machine         11
Microwave                2
Dishwasher               2
Computer Site            2
Name: count, dtype: int64

--------------------------------------------------
House 18
Share of long runs with a dead aggregate: 28.09%
Dominant channel per long run:


Fridge-Freezer           415
Fridge (Garage)          387
Freezer (Garage)         318
Washer Dryer (Garage)     15
Television Site            7
Dishwasher                 4
Washing Machine            3
Desktop Computer           1
Name: count, dtype: int64

--------------------------------------------------


### Findings

All five houses with the most long issue runs are kept. The checks above show the causes are either dead aggregate stretches, which are removed as missing data, or mismatches that the target-only rule and tolerance absorb.

| House | Main cause | Handling |
|---|---|---|
| 3 | Dead aggregate (46% of long runs), plus some calibration offset | Dead stretches are removed |
| 5 | Load-dependent mismatch (about 10%) | Rarely triggers the target-only rule. Note the bias as a limitation; no rescaling. |
| 7 | Tumble dryer cycles (non-target) | Keep. The target-only rule ignores the dryer, so the data comes back. Note a possible aggregate bias at heavy load. |
| 13 | Tumble dryer monitor offset at idle (non-target) | Keep. The target-only rule recovers almost everything. |
| 18 | Dead aggregate (28%), plus possibly unmetered garage fridges or freezers | Keep, with dead stretches removed. Use it for the fridge target only if the step test shows the fridge is visible in the aggregate; otherwise exclude house 18 for the fridge only. |

**Limitations to note:** load-dependent aggregate bias in houses 5 and 7, and the open question of fridge visibility in house 18.